In [1]:
from ultralytics import YOLO
from pathlib import Path
import os
import shutil
from tqdm import tqdm

In [3]:
TRAIN_ROOT = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-train\VisDrone2019-MOT-train"
)

VAL_ROOT = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val"
)


OUTPUT = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\finetune_output"
)


OUTPUT.mkdir(
    exist_ok=True
)

In [4]:
for split in ["train","val"]:

    (OUTPUT/"images"/split).mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT/"labels"/split).mkdir(
        parents=True,
        exist_ok=True
    )

In [5]:
def convert_split(
    root,
    split
):


    seq_path = root/"sequences"
    ann_path = root/"annotations"



    for seq in tqdm(
        list(seq_path.iterdir())
    ):


        # read annotations

        annotations={}


        with open(
            ann_path/(seq.name+".txt")
        ) as f:


            for line in f:


                d=line.strip().split(",")


                frame=int(d[0])

                cls=int(d[7])


                if cls not in [1,2]:
                    continue



                if frame not in annotations:
                    annotations[frame]=[]



                annotations[frame].append(
                    d
                )



        # images

        imgs=sorted(
            seq.glob("*.jpg")
        )



        for img in imgs:


            frame_id=int(
                img.stem
            )


            import cv2

            im=cv2.imread(str(img))

            h,w,_=im.shape



            new_name = (
                seq.name
                +"_"
                +img.name
            )



            shutil.copy(
                img,
                OUTPUT/"images"/split/new_name
            )



            label_file = (

                OUTPUT/
                "labels"/
                split/
                new_name.replace(
                    ".jpg",
                    ".txt"
                )

            )



            with open(label_file,"w") as f:


                for box in annotations.get(
                    frame_id,
                    []
                ):


                    x=float(box[2])
                    y=float(box[3])

                    bw=float(box[4])
                    bh=float(box[5])



                    xc=(x+bw/2)/w
                    yc=(y+bh/2)/h


                    bw=bw/w
                    bh=bh/h



                    f.write(

                    f"0 {xc} {yc} {bw} {bh}\n"

                    )

In [6]:
convert_split(
    TRAIN_ROOT,
    "train"
)


convert_split(
    VAL_ROOT,
    "val"
)

100%|██████████| 7/7 [01:25<00:00, 12.20s/it]


In [23]:
yaml_text = """

path: C:/Users/VAMSEEKRISHNA.P/Desktop/assignment/finetune_output

train: images/train
val: images/val

names:
  0: person

"""


with open(
    "visdrone_person.yaml",
    "w"
) as f:

    f.write(yaml_text)


print(open("visdrone_person.yaml").read())



path: C:/Users/VAMSEEKRISHNA.P/Desktop/assignment/finetune_output

train: images/train
val: images/val

names:
  0: person




In [24]:
from pathlib import Path

root = Path(
    "C:/Users/VAMSEEKRISHNA.P/Desktop/assignment/finetune_output"
)

print(
    "Train images:",
    len(list((root/"images/train").glob("*.jpg")))
)

print(
    "Val images:",
    len(list((root/"images/val").glob("*.jpg")))
)

print(
    "Train labels:",
    len(list((root/"labels/train").glob("*.txt")))
)

print(
    "Val labels:",
    len(list((root/"labels/val").glob("*.txt")))
)

Train images: 24201
Val images: 2846
Train labels: 24201
Val labels: 2846


In [19]:
from ultralytics import YOLO
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4050 Laptop GPU


In [20]:
model = YOLO(
    "yolo11s.pt"
)

model.to("cuda")

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

In [ ]:
results = model.train(


    # dataset
    data="visdrone_person.yaml",


    # training length
    epochs=30,

    patience=5,        # early stopping


    # resolution selected from experiments
    imgsz=1280,


    # RTX4050 safe
    batch=6,


    device=0,

    workers=4,


    # transfer learning
    pretrained=True,

    freeze=10,


    # optimizer
    optimizer="AdamW",

    lr0=1e-4,

    weight_decay=0.0005,


    # scheduler
    cos_lr=True,


    # mixed precision
    amp=True,


    # speed
    cache=True,


    # augmentations
    mosaic=1.0,

    close_mosaic=5,

    scale=0.5,

    translate=0.1,

    fliplr=0.5,


    # disable useless aerial aug
    flipud=0.0,


    # save
    save=True,

    save_period=5,


    project="runs",

    name="YOLO11s_VisDrone_1280"

)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=visdrone_person.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

KeyboardInterrupt: 